In [38]:
import pandas as pd
from collections import defaultdict

In [39]:
matches = pd.read_csv('../data/processed/features_v3.csv')

matches = matches.sort_values('MatchDateTime').reset_index(drop=True)

In [40]:
update_size = 20
home_advantage = 65

In [41]:
def new_rating(old_rating, actual_score, expected_score):
    return old_rating + update_size * (actual_score - expected_score)

In [42]:
def expected_result(r_home, r_away):
    return 1 / (1 + 10**(-(r_home + home_advantage - r_away) / 400))

In [43]:
elos = {}

In [44]:
def get_elo(team):
    if team not in elos:
        elos[team] = 1500
    return elos[team]

In [45]:
home_elo_feature = []
away_elo_feature = []
elo_diff_feature = []
for _, current_match in matches.iterrows():

    home_team = current_match['HomeTeam']
    away_team = current_match['AwayTeam']
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)
    home_elo_feature.append(home_elo)
    away_elo_feature.append(away_elo)
    elo_diff_feature.append(home_elo - away_elo)
    expected = expected_result(home_elo, away_elo)
    if current_match['FTR'] == 'H':
        new_home_elo = new_rating(home_elo, 1, expected)
        new_away_elo = new_rating(away_elo, 0, 1 - expected)
    elif current_match['FTR'] == 'D':
        new_home_elo = new_rating(home_elo, 0.5, expected)
        new_away_elo = new_rating(away_elo, 0.5, 1 - expected)
    else:
        new_home_elo = new_rating(home_elo, 0, expected)
        new_away_elo = new_rating(away_elo, 1, 1 - expected)
    elos[home_team] = new_home_elo
    elos[away_team] = new_away_elo



In [49]:
matches['HomeElo'] = home_elo_feature
matches['AwayElo'] = away_elo_feature
matches['EloDiff'] = elo_diff_feature

C:\Users\harry\AppData\Local\Temp\ipykernel_12916\3788711547.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['EloDiff'] = elo_diff_feature


In [50]:
matches[1000:1020]

,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,HomeAwayGoalsAgainstDiffLast5,GoalDiffLast5,PointsDiffLast5,GoalAgainstDiffLast5,ShotDiffLast5,ShotOTDiffLast5,HomeElo,AwayElo,EloDiffFeature,EloDiff
1000,23-24,E0,17/02/2024,15:00,Fulham,Aston Villa,1,2,A,0,...,0.4,-0.6,1,0.4,-1.6,-2.2,1486.842420,1588.538150,-101.695730,-101.695730
1001,23-24,E0,17/02/2024,15:00,Newcastle,Bournemouth,2,2,D,0,...,-0.2,2.0,5,-0.4,-4.2,1.8,1574.089405,1468.674309,105.415096,105.415096
1002,23-24,E0,17/02/2024,15:00,Nott'm Forest,West Ham,2,0,H,1,...,0.0,1.0,1,0.4,-1.6,0.6,1432.968291,1507.757569,-74.789278,-74.789278
1003,23-24,E0,17/02/2024,15:00,Tottenham,Wolves,1,2,A,0,...,0.4,0.4,4,0.0,1.0,0.2,1602.119085,1486.938116,115.180969,115.180969
1004,23-24,E0,17/02/2024,17:30,Man City,Chelsea,1,1,D,0,...,1.0,0.6,6,1.4,8.2,2.4,1731.086453,1512.638424,218.448029,218.448029
1005,23-24,E0,18/02/2024,14:00,Sheffield United,Brighton,0,5,A,0,...,-0.6,0.4,-1,-1.2,-2.0,-0.4,1417.615518,1544.978651,-127.363133,-127.363133
1006,23-24,E0,18/02/2024,16:30,Luton,Man United,1,2,A,1,...,0.0,0.0,-5,-0.6,1.8,1.4,1456.315262,1604.297067,-147.981805,-147.981805
1007,23-24,E0,19/02/2024,20:00,Everton,Crystal Palace,1,1,D,0,...,1.8,-1.2,-3,1.6,0.6,-2.4,1468.358788,1455.472708,12.886081,12.886081
1008,23-24,E0,20/02/2024,19:30,Man City,Brentford,1,0,H,0,...,1.0,0.6,7,1.4,12.6,2.6,1724.358521,1487.908053,236.450467,236.450467
1009,23-24,E0,21/02/2024,19:30,Liverpool,Luton,4,1,H,0,...,1.2,1.0,7,0.8,1.4,1.8,1698.042583,1448.659268,249.383316,249.383316


In [52]:
matches.tail(10)

,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,HomeAwayGoalsAgainstDiffLast5,GoalDiffLast5,PointsDiffLast5,GoalAgainstDiffLast5,ShotDiffLast5,ShotOTDiffLast5,HomeElo,AwayElo,EloDiffFeature,EloDiff
1890,25-26,E0,24/05/2026,16:00,Crystal Palace,Arsenal,1,2,A,0,...,0.4,-0.4,-10,-2.2,-0.4,0.0,1522.895843,1734.725933,-211.830090,-211.830090
1891,25-26,E0,24/05/2026,16:00,Brighton,Man United,0,3,A,0,...,0.2,0.0,-6,-0.2,-0.4,1.6,1571.127344,1615.428126,-44.300783,-44.300783
1892,25-26,E0,24/05/2026,16:00,Burnley,Wolves,1,1,D,0,...,0.8,0.4,-1,-0.4,-2.4,-0.4,1354.142460,1399.242217,-45.099757,-45.099757
1893,25-26,E0,24/05/2026,16:00,Liverpool,Brentford,1,1,D,0,...,0.6,0.8,2,-0.6,0.6,0.6,1634.653557,1547.639587,87.013970,87.013970
1894,25-26,E0,24/05/2026,16:00,Tottenham,Everton,1,0,H,1,...,-0.6,-0.2,6,1.2,0.2,-0.8,1455.740187,1531.177949,-75.437761,-75.437761
1895,25-26,E0,24/05/2026,16:00,Nott'm Forest,Bournemouth,1,1,D,1,...,-0.4,1.2,-1,-0.4,-2.2,0.4,1527.034544,1594.621120,-67.586576,-67.586576
1896,25-26,E0,24/05/2026,16:00,Sunderland,Chelsea,2,1,H,1,...,0.2,0.6,1,-0.4,-1.4,1.0,1531.035801,1567.100138,-36.064337,-36.064337
1897,25-26,E0,24/05/2026,16:00,Man City,Aston Villa,1,2,A,1,...,1.0,0.0,4,1.2,8.2,1.4,1732.549912,1599.699469,132.850444,132.850444
1898,25-26,E0,24/05/2026,16:00,Fulham,Newcastle,2,0,H,1,...,0.4,-1.2,-2,0.2,-1.6,-2.2,1524.451645,1553.759228,-29.307583,-29.307583
1899,25-26,E0,24/05/2026,16:00,West Ham,Leeds,3,0,H,0,...,0.4,-1.4,-7,-0.8,-1.4,0.0,1473.084440,1502.402041,-29.317600,-29.317600


In [51]:
matches.to_csv('../data/processed/features_v4.csv', index=False)

print('Saved features_v4.csv')

Saved features_v4.csv
